# 22 - Original Language Prediction with Logistic Regression

## 0. Setup

In [ ]:
import time
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)

FEATURES = "../../data/features/11_language_features.csv"
RANDOM_STATE = 42

## 1. Load data and keep the top 3 languages

In [ ]:
df = pd.read_csv(FEATURES)
print("all movies:", df.shape)

top3 = df["target_original_language"].value_counts().index[:3].tolist()
df = df[df["target_original_language"].isin(top3)].reset_index(drop=True)
print("top 3 languages:", top3)
print("kept movies:", df.shape)
df["target_original_language"].value_counts()

## 2. Choose predictors

In [ ]:
num_cols = [
    "release_year", "release_month", "release_decade",
    "log_budget", "log_revenue", "runtime", "log_popularity",
    "genre_count", "company_count", "country_count",
    "cast_size", "crew_size", "keyword_count",
    "title_word_count", "overview_word_count",
]
bin_cols = [
    "has_collection", "has_director", "has_tagline",
    "budget_known", "revenue_known",
]
list_cols = [
    "genres", "production_companies", "production_countries",
    "director", "top_cast", "keywords", "status",
]
text_cols = ["overview", "title"]

## 3. Encode the target and split

In [ ]:
X = df[num_cols + bin_cols + list_cols + text_cols].copy()
X[list_cols + text_cols] = X[list_cols + text_cols].fillna("")

le = LabelEncoder()
y = le.fit_transform(df["target_original_language"])
print(dict(zip(le.classes_, le.transform(le.classes_))))

X_rest, X_test, y_rest, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(
    X_rest, y_rest, test_size=0.25, stratify=y_rest, random_state=RANDOM_STATE)
print("train:", X_train.shape, "val:", X_val.shape, "test:", X_test.shape)

## 4. Baseline

In [ ]:
majority = np.bincount(y_train).argmax()
baseline_acc = (y_val == majority).mean()
print("majority class:", le.classes_[majority])
print("majority baseline accuracy (validation):", round(baseline_acc, 3))

## 5. Turn the columns into numbers

In [ ]:
def split_on_pipe(s):
    return s.split("|")

def one_list_column(col):
    return (col, CountVectorizer(tokenizer=split_on_pipe, lowercase=False,
                                 token_pattern=None, min_df=5, binary=True), col)

def one_text_column(col):
    return (col, TfidfVectorizer(max_features=5000, stop_words="english", min_df=5), col)

preprocess = ColumnTransformer(
    [("numbers", Pipeline([("fill", SimpleImputer(strategy="median")),
                           ("scale", StandardScaler())]), num_cols),
     ("flags", "passthrough", bin_cols)]
    + [one_list_column(c) for c in list_cols]
    + [one_text_column(c) for c in text_cols]
)

def make_model(C=1.0, class_weight=None, solver="lbfgs"):
    return Pipeline([
        ("prep", preprocess),
        ("clf", LogisticRegression(C=C, class_weight=class_weight,
                                   solver=solver, max_iter=1000)),
    ])

## 6. Train multiple models and compare them on the validation set

In [ ]:
grid = list(product([0.01, 0.1, 1, 10, 100], [None, "balanced"], ["lbfgs", "saga"]))
rows = []
for C, cw, solver in grid:
    t = time.time()
    m = make_model(C, cw, solver).fit(X_train, y_train)
    pred = m.predict(X_val)
    rows.append({
        "C": C,
        "class_weight": "none" if cw is None else cw,
        "solver": solver,
        "accuracy": accuracy_score(y_val, pred),
        "precision": precision_score(y_val, pred, average="macro", zero_division=0),
        "recall": recall_score(y_val, pred, average="macro", zero_division=0),
        "f1": f1_score(y_val, pred, average="macro"),
        "seconds": time.time() - t,
    })

val_results = pd.DataFrame(rows).sort_values("f1", ascending=False).reset_index(drop=True)
val_results.round(3)

## 7. Select the best configuration (validation only)

In [ ]:
best = val_results.iloc[0]
best_C = float(best["C"])
best_cw = None if best["class_weight"] == "none" else best["class_weight"]
best_solver = best["solver"]
print("best configuration: C=%s, class_weight=%s, solver=%s" % (best_C, best_cw, best_solver))
print("validation accuracy=%.3f precision=%.3f recall=%.3f f1=%.3f" % (
    best["accuracy"], best["precision"], best["recall"], best["f1"]))

## 8. Compare configurations on the validation set

In [ ]:
labels = [f"C={r.C:g} | {r.class_weight} | {r.solver}" for r in val_results.itertuples()]
ypos = np.arange(len(val_results))

fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(ypos - 0.2, val_results["f1"], height=0.4, label="macro-F1")
ax.barh(ypos + 0.2, val_results["accuracy"], height=0.4, label="accuracy")
ax.set_yticks(ypos, labels, fontsize=8)
ax.invert_yaxis()
ax.set_xlim(val_results["f1"].min() - 0.05, 1)
ax.set_xlabel("validation score")
ax.set_title("Validation performance of each configuration")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 9. Final evaluation on the test set

In [ ]:
model = make_model(best_C, best_cw, best_solver).fit(X_rest, y_rest)
y_pred = model.predict(X_test)

print("test accuracy:  %.3f" % accuracy_score(y_test, y_pred))
print("test precision: %.3f (macro)" % precision_score(y_test, y_pred, average="macro"))
print("test recall:    %.3f (macro)" % recall_score(y_test, y_pred, average="macro"))
print("test F1:        %.3f (macro)" % f1_score(y_test, y_pred, average="macro"))
print("baseline accuracy: %.3f" % (y_test == majority).mean())
print()
print(classification_report(y_test, y_pred, target_names=le.classes_))

## 10. Confusion matrix (test set)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(le.classes_)), le.classes_)
ax.set_yticks(range(len(le.classes_)), le.classes_)
ax.set_xlabel("predicted language")
ax.set_ylabel("actual language")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im)
ax.set_title("Confusion matrix (test set)")
plt.tight_layout()
plt.show()

## 11. F1-score by language (test set)

In [ ]:
f1_by_class = f1_score(y_test, y_pred, average=None)
fig, ax = plt.subplots(figsize=(5.5, 4))
bars = ax.bar(le.classes_, f1_by_class)
ax.bar_label(bars, fmt="%.3f")
ax.set_ylim(0, 1.05)
ax.set_xlabel("language")
ax.set_ylabel("F1-score")
ax.set_title("F1-score by language (test set)")
plt.tight_layout()
plt.show()

## 12. Inference example

In [ ]:
sample = X_test.iloc[[0]]
true_lang = le.inverse_transform([y_test[0]])[0]

pred_lang = le.inverse_transform(model.predict(sample))[0]
probs = model.predict_proba(sample)[0]

print("INPUT")
print("  title:               ", sample["title"].iloc[0])
print("  release year:        ", sample["release_year"].iloc[0])
print("  genres:              ", sample["genres"].iloc[0])
print("  production countries:", sample["production_countries"].iloc[0])
print("  top cast:            ", sample["top_cast"].iloc[0])
print("  overview:            ", sample["overview"].iloc[0][:150], "...")
print()
print("OUTPUT")
print("  predicted language:", pred_lang)
print("  actual language:   ", true_lang)
print("  probabilities:     ", {c: round(float(p), 3) for c, p in zip(le.classes_, probs)})

## 13. Which languages are easy and hard to predict